<a href="https://colab.research.google.com/github/Aman21adu/aastu-week2-semantic-search/blob/main/Week2_Mini_Semantic_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AASTU AWS AI/ML — Week 2: Mini Semantic Search

**Final lab notebook | FAQs + pandas + text cleaning + chunking + embeddings + ChromaDB**

**Required task:** Load a CSV of FAQs → clean the text → split long text into chunks → encode each chunk into an embedding → store embeddings in Chroma → ask a question and print the **top 3 matching FAQs**.

**No API key needed.** Uses the `sentence-transformers/all-MiniLM-L6-v2` model, following our course practical. The first run requires an internet connection to install packages and download the model. The sample FAQs are illustrative; they are **not official AASTU policy**.

**Run in Colab:** `Runtime → Run all`. The notebook creates the sample `faqs.csv` automatically if that file is not present. If Colab asks you to restart after installing packages, restart and run all again.

## 1. Install and import libraries

- **pandas** loads FAQ data from a CSV file.
- **Sentence Transformers** converts text into numerical embeddings.
- **ChromaDB** stores the embeddings and retrieves similar text.

In [ ]:
%pip install -q "pandas>=2.0,<3" "numpy>=1.26,<3" "sentence-transformers>=3.0,<6" "chromadb>=1.0,<2"

In [2]:
import re
from pathlib import Path
import pandas as pd
from IPython.display import display

print('pandas version:', pd.__version__)

pandas version: 2.2.3


## 2. Load the FAQ CSV using pandas

The GitHub repository includes `faqs.csv`. If you open the notebook directly from GitHub in **Google Colab**, that file is *not automatically uploaded* to the runtime. To make the assignment runnable in one click, this cell recreates **the same CSV** if needed, and then loads it with `pd.read_csv()`.

The sample intentionally includes one duplicate and one empty answer, giving us something to clean.

In [3]:
CSV_PATH = Path('faqs.csv')
SAMPLE_FAQS = [{'id': 'faq_001', 'question': 'How do I reset my password?', 'answer': 'Open the sign-in page and choose Forgot Password. Follow the instructions sent to your registered email.', 'category': 'account'}, {'id': 'faq_002', 'question': 'I cannot log in to my account.', 'answer': 'Check that your email and password are correct. If you still cannot sign in, reset your password.', 'category': 'account'}, {'id': 'faq_003', 'question': 'When does course registration open?', 'answer': 'Course registration dates are announced before the start of each semester in the student portal.', 'category': 'registration'}, {'id': 'faq_004', 'question': 'How can I contact the registrar?', 'answer': 'Contact the registrar office through the official student support channel during working hours.', 'category': 'support'}, {'id': 'faq_005', 'question': 'Where can I see my grades?', 'answer': 'Sign in to the student portal and open the academic results or grades section.', 'category': 'academics'}, {'id': 'faq_006', 'question': 'How do I pay my tuition?', 'answer': 'Follow the payment instructions published by the institution and keep your payment receipt.', 'category': 'finance'}, {'id': 'faq_007', 'question': 'I need help accessing the online learning platform.', 'answer': 'Verify your account details and internet connection. Contact support if the platform remains unavailable.', 'category': 'support'}, {'id': 'faq_008', 'question': 'How do I download my academic transcript?', 'answer': 'Check the student portal for transcript services or ask the registrar office about the official process.', 'category': 'academics'}, {'id': 'faq_009', 'question': 'How do I connect to campus Wi-Fi?', 'answer': 'Ask campus IT support for the current Wi-Fi network name and login instructions. Use your authorized student credentials when required.', 'category': 'technology'}, {'id': 'faq_010', 'question': 'Where do I find library e-books?', 'answer': 'Open the university library website and look for electronic resources. Ask a librarian if access to a database fails.', 'category': 'library'}, {'id': 'faq_011', 'question': 'How can I check my class timetable?', 'answer': 'Open the registered courses or timetable section of the student portal. Contact your department about timetable conflicts.', 'category': 'academics'}, {'id': 'faq_012', 'question': 'Where can I learn about scholarships?', 'answer': 'Review scholarship announcements through official university channels. Confirm eligibility, required documents and deadlines before applying.', 'category': 'finance'}, {'id': 'faq_013', 'question': 'How do I get academic advising?', 'answer': 'First check your course plan and identify the subjects you are uncertain about. Prepare your current transcript, any prerequisites, and questions about next-semester registration. Then request an academic advising appointment using the official university process. Explain schedule conflicts or academic concerns clearly. Ask the advisor to confirm the next steps and any required approvals. Keep a note of the recommendations so you can follow up later.', 'category': 'academics'}, {'id': 'faq_014', 'question': 'What should I do if I lose my student ID card?', 'answer': 'Report the loss to the appropriate campus office. Ask about replacement requirements, identity verification and any applicable fee.', 'category': 'campus'}, {'id': 'faq_015', 'question': 'Can I use university computer labs?', 'answer': 'Check the published lab schedule and usage rules. Ask your department or lab administrator about access permissions.', 'category': 'technology'}, {'id': 'faq_016', 'question': 'How can I join student clubs?', 'answer': 'Look for student-activity announcements or contact the campus student affairs office for currently active clubs and registration steps.', 'category': 'campus'}, {'id': 'faq_003', 'question': 'When does course registration open?', 'answer': 'Course registration dates are announced before the start of each semester in the student portal.', 'category': 'registration'}, {'id': 'faq_017', 'question': 'Can I request an official enrollment letter?', 'answer': '', 'category': 'academics'}]

if not CSV_PATH.exists():
    pd.DataFrame(SAMPLE_FAQS).to_csv(CSV_PATH, index=False)
    print('Created sample CSV for this Colab runtime.')

raw_faqs = pd.read_csv(CSV_PATH)
print('Loaded raw rows:', len(raw_faqs))
display(raw_faqs.head())

Created sample CSV for this Colab runtime.
Loaded raw rows: 18


,id,question,answer,category
0,faq_001,How do I reset my password?,Open the sign-in page and choose Forgot Passwo...,account
1,faq_002,I cannot log in to my account.,Check that your email and password are correct...,account
2,faq_003,When does course registration open?,Course registration dates are announced before...,registration
3,faq_004,How can I contact the registrar?,Contact the registrar office through the offic...,support
4,faq_005,Where can I see my grades?,Sign in to the student portal and open the aca...,academics


## 3. Clean FAQ rows with functions

Real-world data may contain blank answers, repeated rows and inconsistent whitespace. We define small reusable functions for each step and keep only FAQs with a valid question and answer.

In [4]:
def standardize_columns(frame):
    cleaned = frame.copy()
    for col in ('id', 'question', 'answer', 'category'):
        cleaned[col] = cleaned[col].astype('string').str.strip()
        cleaned[col] = cleaned[col].str.replace(r'\s+', ' ', regex=True)
    return cleaned

def remove_missing_faqs(frame):
    cleaned = frame.dropna(subset=['id', 'question', 'answer', 'category']).copy()
    return cleaned[(cleaned['question'] != '') & (cleaned['answer'] != '')]

def remove_duplicate_faqs(frame):
    return frame.drop_duplicates(subset=['question', 'answer', 'category']).reset_index(drop=True)

faqs = remove_duplicate_faqs(remove_missing_faqs(standardize_columns(raw_faqs)))
assert faqs['id'].is_unique, 'FAQ IDs should be unique.'
assert not faqs[['question','answer']].isna().any().any()
print(f'Raw FAQs: {len(raw_faqs)} | Clean FAQs: {len(faqs)} | Removed: {len(raw_faqs)-len(faqs)}')
display(faqs[['id','question','category']].head(7))

Raw FAQs: 18 | Clean FAQs: 16 | Removed: 2


,id,question,category
0,faq_001,How do I reset my password?,account
1,faq_002,I cannot log in to my account.,account
2,faq_003,When does course registration open?,registration
3,faq_004,How can I contact the registrar?,support
4,faq_005,Where can I see my grades?,academics
5,faq_006,How do I pay my tuition?,finance
6,faq_007,I need help accessing the online learning plat...,support


## 4. Normalize text and split long FAQ text into chunks

**Text cleaning** removes excess whitespace and standardizes capitalization. **Chunking** splits a long passage into overlapping word groups, preserving some context between chunks.

We embed the *question plus its answer*. Every chunk has metadata pointing back to its original FAQ, which will let us return three **distinct** FAQ matches rather than repeating one long answer.

In [5]:
def normalize_text(text):
    text = str(text).lower().strip()
    return re.sub(r'\s+', ' ', text)

def chunk_text(text, chunk_size=45, overlap=8):
    if chunk_size <= 0 or not 0 <= overlap < chunk_size:
        raise ValueError('chunk_size must be > 0; overlap must be smaller than chunk_size.')
    words = text.split()
    if not words:
        return []
    step = chunk_size - overlap
    pieces = []
    for start in range(0, len(words), step):
        pieces.append(' '.join(words[start:start + chunk_size]))
        if start + chunk_size >= len(words):
            break
    return pieces

chunks = []
for _, faq in faqs.iterrows():
    full_text = normalize_text(f"Question: {faq['question']} Answer: {faq['answer']}")
    for chunk_no, passage in enumerate(chunk_text(full_text)):
        chunks.append({
            'chunk_id': f"{faq['id']}_chunk_{chunk_no}",
            'faq_id': str(faq['id']),
            'question': str(faq['question']),
            'answer': str(faq['answer']),
            'category': str(faq['category']),
            'chunk_no': int(chunk_no),
            'text': passage,
        })

chunks_df = pd.DataFrame(chunks)
print(f'Clean FAQs: {len(faqs)} | Searchable text chunks: {len(chunks_df)}')
display(chunks_df[['chunk_id', 'category', 'text']].head(4))

Clean FAQs: 16 | Searchable text chunks: 17


,chunk_id,category,text
0,faq_001_chunk_0,account,question: how do i reset my password? answer: ...
1,faq_002_chunk_0,account,question: i cannot log in to my account. answe...
2,faq_003_chunk_0,registration,question: when does course registration open? ...
3,faq_004_chunk_0,support,question: how can i contact the registrar? ans...


## 5. Generate text embeddings

A **text embedding** is a vector of numbers encoding aspects of a passage's meaning. We use the *same model* to embed FAQ chunks and user questions.

The model may be downloaded on the first run; this is normal in Colab.

In [6]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(
    chunks_df['text'].tolist(),
    normalize_embeddings=True,
    show_progress_bar=False,
)
print('Embeddings shape (chunks, vector dimensions):', embeddings.shape)
assert embeddings.shape[0] == len(chunks_df)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings shape (chunks, vector dimensions): (17, 384)


## 6. Store embeddings in ChromaDB

**Chroma** is a vector database: it stores the text passage, embedding, and metadata. We explicitly supply our MiniLM embeddings, so Chroma does not need to embed these texts again. A persistent local collection is created for the lab.

For repeatability, we reset **only** our Week 2 demo collection when this cell is rerun. Do not use that reset strategy with important real data.

In [7]:
import chromadb

client = chromadb.PersistentClient(path='./chroma_faq_db')
COLLECTION_NAME = 'aastu_week2_faq_search'

# Start with a clean demo collection when rerunning the notebook.
if COLLECTION_NAME in [c.name for c in client.list_collections()]:
    client.delete_collection(COLLECTION_NAME)
collection = client.create_collection(name=COLLECTION_NAME)

collection.add(
    ids=chunks_df['chunk_id'].tolist(),
    documents=chunks_df['text'].tolist(),
    embeddings=embeddings.tolist(),
    metadatas=[{
        'faq_id': row['faq_id'],
        'question': row['question'],
        'answer': row['answer'],
        'category': row['category'],
        'chunk_no': row['chunk_no'],
    } for row in chunks],
)

print('Stored passages in Chroma:', collection.count())
assert collection.count() == len(chunks_df)

Stored passages in Chroma: 17


## 7. Query the database and show the top 3 matching FAQs

For each user question, we create a query embedding and ask Chroma for the closest stored chunks. We deduplicate by FAQ ID so that the result contains **three different FAQs**. The returned **distance** is a similarity-related distance, **not a confidence percentage**; smaller is generally closer.

In [8]:
def search_faq(query, top_k=3):
    query = str(query).strip()
    if not query:
        raise ValueError('Question must not be empty.')
    if top_k < 1:
        raise ValueError('top_k must be >= 1')

    query_vector = model.encode([normalize_text(query)], normalize_embeddings=True)[0]
    # Small teaching dataset: fetching all chunks ensures enough distinct FAQs.
    hits = collection.query(
        query_embeddings=[query_vector.tolist()],
        n_results=collection.count(),
        include=['metadatas', 'distances'],
    )

    matches, seen_faqs = [], set()
    for meta, distance in zip(hits['metadatas'][0], hits['distances'][0]):
        if meta['faq_id'] in seen_faqs:
            continue
        seen_faqs.add(meta['faq_id'])
        matches.append({
            'rank': len(matches) + 1,
            'question': meta['question'],
            'answer': meta['answer'],
            'category': meta['category'],
            'distance': round(float(distance), 4),
        })
        if len(matches) == top_k:
            break
    return pd.DataFrame(matches)

query = 'I forgot my password and cannot log in'
print('USER QUERY:', query)
results = search_faq(query)
display(results)
assert len(results) == 3, 'Expected three different matches.'

USER QUERY: I forgot my password and cannot log in


,rank,question,answer,category,distance
0,1,How do I reset my password?,Open the sign-in page and choose Forgot Passwo...,account,0.6328
1,2,I cannot log in to my account.,Check that your email and password are correct...,account,0.6739
2,3,I need help accessing the online learning plat...,Verify your account details and internet conne...,support,1.4257


## 8. Test several example queries

Try paraphrases (not exactly copied FAQ questions) to demonstrate **semantic** rather than simple keyword search. Every query prints the three closest FAQ questions, their associated answers, and distances.

In [9]:
example_queries = [
    'I forgot how to sign in to my student account',
    'Where can I check my examination marks?',
    'I need a copy of my academic transcript',
    'How can my computer join the campus wireless network?',
    'I need help picking subjects for next semester',
]

for q in example_queries:
    print('\n' + '=' * 65)
    print('QUESTION:', q)
    top_matches = search_faq(q, top_k=3)
    assert len(top_matches) == 3
    display(top_matches)

print('\nSUCCESS: Printed top 3 matches for every example query.')


QUESTION: I forgot how to sign in to my student account


,rank,question,answer,category,distance
0,1,How do I reset my password?,Open the sign-in page and choose Forgot Passwo...,account,0.9069
1,2,I cannot log in to my account.,Check that your email and password are correct...,account,0.9613
2,3,Where can I see my grades?,Sign in to the student portal and open the aca...,academics,0.9664



QUESTION: Where can I check my examination marks?


,rank,question,answer,category,distance
0,1,Where can I see my grades?,Sign in to the student portal and open the aca...,academics,0.7331
1,2,How can I check my class timetable?,Open the registered courses or timetable secti...,academics,1.0702
2,3,How do I download my academic transcript?,Check the student portal for transcript servic...,academics,1.2886



QUESTION: I need a copy of my academic transcript


...


QUESTION: How can my computer join the campus wireless network?


,rank,question,answer,category,distance
0,1,How do I connect to campus Wi-Fi?,Ask campus IT support for the current Wi-Fi ne...,technology,0.7053
1,2,How can I join student clubs?,Look for student-activity announcements or con...,campus,1.2419
2,3,Can I use university computer labs?,Check the published lab schedule and usage rul...,technology,1.3417



QUESTION: I need help picking subjects for next semester


,rank,question,answer,category,distance
0,1,Where can I see my grades?,Sign in to the student portal and open the aca...,academics,1.3025
1,2,How can I check my class timetable?,Open the registered courses or timetable secti...,academics,1.3771
2,3,How do I get academic advising?,First check your course plan and identify the ...,academics,1.5092



SUCCESS: Printed top 3 matches for every example query.


## 9. Try your own question (optional)

Change the text in the cell below, then run just this cell. The returned results are related passages, not guaranteed correct answers. For out-of-scope questions, the nearest FAQ may still be irrelevant.

In [10]:
my_question = 'How do I get academic advising?'
display(search_faq(my_question, top_k=3))

,rank,question,answer,category,distance
0,1,How do I get academic advising?,First check your course plan and identify the ...,academics,0.3866
1,2,Where can I learn about scholarships?,Review scholarship announcements through offic...,finance,1.1470
2,3,Where can I see my grades?,Sign in to the student portal and open the aca...,academics,1.4060


## 10. Conclusion / limitations

**We completed the Week 2 assignment:**

1. **Loaded** an FAQ CSV using `pd.read_csv()`.
2. **Cleaned** missing, duplicate, and irregular text records.
3. **Chunked** the question-and-answer text into overlapping passages.
4. **Generated embeddings** with `all-MiniLM-L6-v2`.
5. **Stored vectors and metadata** in ChromaDB.
6. **Queried** Chroma and displayed the **top 3 different FAQ matches** per question.

**Limitations:** FAQs here are illustrative, not official policy. Similarity is not guaranteed correctness. A production system needs a verified knowledge base, retrieval evaluation, and a no-answer mechanism for unrelated questions.

**Submission:** Save this notebook **with output cells visible** after `Runtime → Run all`, upload the executed `.ipynb` to GitHub, and share a working Colab link.